In [7]:
import pandas as pd
import os

META_PATH = "../data/metadata/GSE135779_sample_info.csv"
RAW_DIR = "../data/raw/GSE135779"
os.makedirs(RAW_DIR, exist_ok=True)

# To Read metadata
df = pd.read_csv(META_PATH)
print(f"Total samples: {len(df)}")
print(df.head())
# To Parse the characteristics column
def parse_characteristics(char_str):
    parts = [p.strip() for p in char_str.split("|")]
    info = {}
    for part in parts:
        if ":" in part:
            key, val = part.split(":", 1)
            info[key.strip()] = val.strip()
    return info

# Apply parsing
parsed = df["characteristics"].apply(parse_characteristics)
df["age"] = parsed.apply(lambda x: x.get("age", "unknown"))
df["age_group"] = parsed.apply(lambda x: x.get("age group", "unknown"))
df["group"] = parsed.apply(lambda x: x.get("groups", "unknown"))

# Show counts
print("Sample counts by group:")
print(df.groupby(["age_group", "group"]).size())

def make_category(row):
    ag = row["age_group"]
    g = row["group"]
    if ag == "Children" and g == "SLE":
        return "cSLE"
    elif ag == "Children" and g == "HD":
        return "cHD"
    elif ag == "Adult" and g == "SLE":
        return "aSLE"
    elif ag == "Adult" and g == "HD":
        return "aHD"
    return "unknown"

df["category"] = df.apply(make_category, axis=1)
print("\nBy category:")
print(df["category"].value_counts())

# Save the annotated metadata
df.to_csv("../data/metadata/GSE135779_annotated.csv", index=False)
print("\nSaved annotated metadata to data/metadata/GSE135779_annotated.csv")

Total samples: 56
          gsm            title  source_name  \
0  GSM4029896  cSLE1 [JB17001]  Child SLE 1   
1  GSM4029897  cSLE2 [JB17002]  Child SLE 2   
2  GSM4029898  cSLE3 [JB17003]  Child SLE 3   
3  GSM4029899  cSLE4 [JB17004]  Child SLE 4   
4  GSM4029900  cSLE5 [JB17005]  Child SLE 5   

                               characteristics  description  
0  age: 17 | age group: Children | groups: SLE          NaN  
1  age: 18 | age group: Children | groups: SLE          NaN  
2  age: 16 | age group: Children | groups: SLE          NaN  
3  age: 17 | age group: Children | groups: SLE          NaN  
4  age: 14 | age group: Children | groups: SLE          NaN  
Sample counts by group:
age_group  group
Adult      HD        5
           SLE       7
Children   HD       11
           SLE      33
dtype: int64

By category:
category
cSLE    33
cHD     11
aSLE     7
aHD      5
Name: count, dtype: int64

Saved annotated metadata to data/metadata/GSE135779_annotated.csv


In [ ]:
import time
import requests

META_PATH = "../data/metadata/GSE135779_annotated.csv"
RAW_DIR = "../data/raw/GSE135779"
os.makedirs(RAW_DIR, exist_ok=True)

df = pd.read_csv(META_PATH)

def get_supplementary_files(gsm, timeout=30):
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gsm}&targ=self&view=full&form=text"
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    files = {}
    for line in r.text.splitlines():
        if line.startswith("!Sample_supplementary_file"):
            file_url = line.split("=", 1)[1].strip()
            if file_url.startswith("ftp://"):
                file_url = file_url.replace("ftp://", "https://", 1)  # avoid the FTP block
            fname = file_url.rstrip("/").split("/")[-1]
            files[fname] = file_url
    return files

In [6]:
df

,gsm,title,source_name,characteristics,description,age,age_group,group,category
0,GSM4029896,cSLE1 [JB17001],Child SLE 1,age: 17 | age group: Children | groups: SLE,NaN,17,Children,SLE,cSLE
1,GSM4029897,cSLE2 [JB17002],Child SLE 2,age: 18 | age group: Children | groups: SLE,NaN,18,Children,SLE,cSLE
2,GSM4029898,cSLE3 [JB17003],Child SLE 3,age: 16 | age group: Children | groups: SLE,NaN,16,Children,SLE,cSLE
3,GSM4029899,cSLE4 [JB17004],Child SLE 4,age: 17 | age group: Children | groups: SLE,NaN,17,Children,SLE,cSLE
4,GSM4029900,cSLE5 [JB17005],Child SLE 5,age: 14 | age group: Children | groups: SLE,NaN,14,Children,SLE,cSLE
5,GSM4029901,cSLE6 [JB17006],Child SLE 6,age: 16 | age group: Children | groups: SLE,NaN,16,Children,SLE,cSLE
6,GSM4029902,cSLE7 [JB17007],Child SLE 7,age: 17 | age group: Children | groups: SLE,NaN,17,Children,SLE,cSLE
7,GSM4029903,cSLE8 [JB17008],Child SLE 8,age: 16 | age group: Children | groups: SLE,NaN,16,Children,SLE,cSLE
8,GSM4029904,cSLE10 [JB17015],Child SLE 10,age: 18 | age group: Children | groups: SLE,NaN,18,Children,SLE,cSLE
9,GSM4029905,cSLE11 [JB17016],Child SLE 11,age: 17 | age group: Children | groups: SLE,NaN,17,Children,SLE,cSLE


In [ ]:
def download_file(url, out_path, max_retries=3, timeout=60):
    if os.path.exists(out_path):
        size_mb = os.path.getsize(out_path) / (1024 * 1024)
        print(f"  SKIP (exists): {os.path.basename(out_path)} ({size_mb:.1f} MB)")
        return True
    for attempt in range(max_retries):
        try:
            print(f"  DOWNLOAD: {os.path.basename(out_path)} ...")
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with open(out_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            size_mb = os.path.getsize(out_path) / (1024 * 1024)
            print(f"  DONE ({size_mb:.1f} MB)")
            return True
        except Exception as e:
            print(f"  ERROR (attempt {attempt+1}/{max_retries}): {e}")
            time.sleep(2)
    print(f"  FAILED after {max_retries} attempts: {os.path.basename(out_path)}")
    return False

In [ ]:
failed, success, total_files = [], 0, 0

for i, row in df.iterrows():
    gsm, cat = row["gsm"], row["category"]
    print(f"\n[{i+1}/{len(df)}] {gsm} ({cat})")

    try:
        suppl = get_supplementary_files(gsm)
    except Exception as e:
        print(f"  ERROR fetching file list: {e}")
        failed.append({"gsm": gsm, "url": "metadata lookup"})
        continue

    wanted = [f for f in suppl if any(k in f.lower() for k in ("barcodes", "features", "genes", "matrix"))]
    if not wanted:
        print(f"  WARNING: no recognizable 10x files. Files present: {list(suppl.keys())}")

    out_dir = os.path.join(RAW_DIR, gsm)
    os.makedirs(out_dir, exist_ok=True)

    for fname in wanted:
        total_files += 1
        out_path = os.path.join(out_dir, fname)
        ok = download_file(suppl[fname], out_path)
        success += ok
        if not ok:
            failed.append({"gsm": gsm, "url": suppl[fname]})
        time.sleep(0.5)

print(f"\n{'='*60}\nFinished: {success}/{total_files} files downloaded successfully")
if failed:
    print(f"Failed: {len(failed)}")
    for f in failed:
        print(f"  - {f}")
print(f"{'='*60}")


[1/56] GSM4029896 (cSLE)
  DOWNLOAD: GSM4029896_JB17001_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029896_JB17001_matrix.mtx.gz ...
  DONE (14.2 MB)

[2/56] GSM4029897 (cSLE)
  DOWNLOAD: GSM4029897_JB17002_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029897_JB17002_matrix.mtx.gz ...
  DONE (15.2 MB)

[3/56] GSM4029898 (cSLE)
  DOWNLOAD: GSM4029898_JB17003_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029898_JB17003_matrix.mtx.gz ...
  DONE (14.9 MB)

[4/56] GSM4029899 (cSLE)
  DOWNLOAD: GSM4029899_JB17004_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029899_JB17004_matrix.mtx.gz ...
  DONE (20.6 MB)

[5/56] GSM4029900 (cSLE)
  DOWNLOAD: GSM4029900_JB17005_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029900_JB17005_matrix.mtx.gz ...
  DONE (13.4 MB)

[6/56] GSM4029901 (cSLE)
  DOWNLOAD: GSM4029901_JB17006_barcodes.tsv.gz ...
  DONE (0.0 MB)
  DOWNLOAD: GSM4029901_JB17006_matrix.mtx.gz ...
  DONE (17.5 MB)

[7/56] GSM4029902 (cSLE)
  DOWNLOAD: GSM40299

In [ ]:
import glob

all_mtx = sorted(glob.glob(f"{RAW_DIR}/*/GSM*_matrix.mtx.gz"))
all_barcodes = sorted(glob.glob(f"{RAW_DIR}/*/GSM*_barcodes.tsv.gz"))
all_features = sorted(glob.glob(f"{RAW_DIR}/*/GSM*_features.tsv.gz"))

print(f"Matrix files:  {len(all_mtx)} / 56")
print(f"Barcode files: {len(all_barcodes)} / 56")
print(f"Feature files: {len(all_features)} / 56")

sizes = []
for mtx in all_mtx:
    size_mb = os.path.getsize(mtx) / (1024 * 1024)
    gsm = os.path.basename(os.path.dirname(mtx))
    sizes.append((gsm, size_mb))

size_df = pd.DataFrame(sizes, columns=["gsm", "size_mb"])
print(f"\nTotal downloaded: {size_df['size_mb'].sum():.1f} MB ({size_df['size_mb'].sum()/1024:.2f} GB)")
print(f"\nSmallest: {size_df['size_mb'].min():.1f} MB")
print(f"Largest:  {size_df['size_mb'].max():.1f} MB")
print(f"Average:  {size_df['size_mb'].mean():.1f} MB")

small = size_df[size_df["size_mb"] < 5]
if len(small) > 0:
    print(f"\nWARNING: {len(small)} samples have very small matrix files (<5 MB):")
    print(small)

Matrix files:  56 / 56
Barcode files: 56 / 56
Feature files: 0 / 56

Total downloaded: 1238.1 MB (1.21 GB)

Smallest: 9.1 MB
Largest:  45.4 MB
Average:  22.1 MB
